# Mixed Signals Quant Model
### 30-Day Historical Record — LSTM Neural Network
**Output:** `SELL` / `HOLD` / `BUY`

---

## 0. Install dependencies
Run this cell if you're missing any packages.

In [ ]:
#!pip install numpy pandas torch scikit-learn

## 1. Imports & reproducibility

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

np.random.seed(42)
torch.manual_seed(42)

print("Libraries loaded.")
print(f"PyTorch version: {torch.__version__}")

## 2. Define dimensions & feature names

In [ ]:
n_cases   = 1200   #number of behavioral histories
n_days    = 30     #1 month window
n_features = 11    #daily behavior factors

feature_names = [
    "engagement_volume",       #texts per day
    "latency_hours",           #avg response time
    "execution_ratio",         #follow through quality
    "initiative_factor",       #did he initiate
    "sentiment_stability",     #emotional consistency
    "after_hours_activity",    #late night text ratio
    "default_rate",            #cancellations / dropped plans
    "passive_engagement",      #lurking without direct contact
    "disclosure_quality",      #clarity of intent
    "behavioral_beta",         #volatility
    "guidance_delivery_spread" #words vs actions gap
]

print(f"Dataset shape: ({n_cases}, {n_days}, {n_features})")
print(f"Features: {feature_names}")

## 3. Simulate 30-day behavioral sequences
Each case = 30 days × 11 features, drawn from a latent profile type.

In [ ]:
X = np.zeros((n_cases, n_days, n_features))

for i in range(n_cases):
    profile_type = np.random.choice(["buy", "hold", "sell"], p=[0.30, 0.30, 0.40])

    for d in range(n_days):
        if profile_type == "buy":
            vals = [
                np.random.normal(10, 2),
                np.random.normal(3, 1.5),
                np.random.normal(0.85, 0.08),
                np.random.normal(0.70, 0.10),
                np.random.normal(0.82, 0.08),
                np.random.normal(0.15, 0.08),
                np.random.normal(0.08, 0.05),
                np.random.binomial(1, 0.15),
                np.random.normal(4.3, 0.5),
                np.random.normal(1.8, 0.7),
                np.random.normal(0.15, 0.08)
            ]
        elif profile_type == "hold":
            vals = [
                np.random.normal(6, 2.5),
                np.random.normal(15, 8),
                np.random.normal(0.50, 0.15),
                np.random.normal(0.45, 0.15),
                np.random.normal(0.50, 0.15),
                np.random.normal(0.45, 0.15),
                np.random.normal(0.35, 0.15),
                np.random.binomial(1, 0.45),
                np.random.normal(2.7, 0.8),
                np.random.normal(5.0, 1.2),
                np.random.normal(0.45, 0.15)
            ]
        else:  # sell
            vals = [
                np.random.normal(3, 2),
                np.random.normal(38, 15),
                np.random.normal(0.20, 0.10),
                np.random.normal(0.18, 0.10),
                np.random.normal(0.25, 0.12),
                np.random.normal(0.72, 0.15),
                np.random.normal(0.65, 0.15),
                np.random.binomial(1, 0.70),
                np.random.normal(1.2, 0.6),
                np.random.normal(8.0, 1.0),
                np.random.normal(0.75, 0.12)
            ]

        X[i, d, :] = vals

print(f"X shape: {X.shape}")

## 4. Clip features to valid ranges

In [ ]:
X[:, :, 0]  = np.clip(X[:, :, 0],  0, 25)   #engagement_volume
X[:, :, 1]  = np.clip(X[:, :, 1],  0, 120)  #latency_hours
X[:, :, 2]  = np.clip(X[:, :, 2],  0, 1)    #execution_ratio
X[:, :, 3]  = np.clip(X[:, :, 3],  0, 1)    #initiative_factor
X[:, :, 4]  = np.clip(X[:, :, 4],  0, 1)    #sentiment_stability
X[:, :, 5]  = np.clip(X[:, :, 5],  0, 1)    #after_hours_activity
X[:, :, 6]  = np.clip(X[:, :, 6],  0, 1)    #default_rate
X[:, :, 7]  = np.clip(X[:, :, 7],  0, 1)    #passive_engagement
X[:, :, 8]  = np.clip(X[:, :, 8],  0, 5)    #disclosure_quality
X[:, :, 9]  = np.clip(X[:, :, 9],  0, 10)   #behavioral_beta
X[:, :, 10] = np.clip(X[:, :, 10], 0, 1)    #guidance_delivery_spread

print("Clipping complete.")
print(pd.DataFrame(X.mean(axis=(0,1)), index=feature_names, columns=["global mean"]).round(3))

## 5. Label each 30-day sequence
`0 = SELL` · `1 = HOLD` · `2 = BUY`

In [ ]:
def assign_sequence_label(sequence):
    m = sequence.mean(axis=0)
    score  =  m[0]  * 0.08                #engagement_volume
    score -= (m[1] / 120.0) * 2.0         #latency_hours
    score +=  m[2]  * 2.2                 #execution_ratio
    score +=  m[3]  * 1.5                 #initiative_factor
    score +=  m[4]  * 1.8                 #sentiment_stability
    score -=  m[5]  * 1.0                 #after_hours_activity
    score -=  m[6]  * 2.0                 #default_rate
    score -=  m[7]  * 0.8                 #passive_engagement
    score += (m[8] / 5.0) * 1.8           #disclosure_quality
    score -= (m[9] / 10.0) * 1.8          #behavioral_beta
    score -=  m[10] * 2.2                 #guidance_delivery_spread

    if score >= 2.4:
        return 2   #BUY
    elif score >= 0.8:
        return 1   #HOLD
    else:
        return 0   #SELL

y = np.array([assign_sequence_label(X[i]) for i in range(n_cases)])

unique, counts = np.unique(y, return_counts=True)
label_map = {0: "SELL (Red Flag)", 1: "HOLD (Yellow Flag)", 2: "BUY (Green Flag)"}
print("Label distribution:")
for u, c in zip(unique, counts):
    print(f"  {label_map[u]:4s}: {c} ({c/n_cases*100:.1f}%)")

## 6. Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} cases | Test: {X_test.shape[0]} cases")

## 7. Standardise features

In [ ]:
scaler = StandardScaler()

X_train_2d = X_train.reshape(-1, n_features)
X_test_2d  = X_test.reshape(-1, n_features)

X_train_scaled = scaler.fit_transform(X_train_2d).reshape(-1, n_days, n_features)
X_test_scaled  = scaler.transform(X_test_2d).reshape(-1, n_days, n_features)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_scaled,  dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.long)

print("Tensors ready.")
print(f"  X_train_tensor: {X_train_tensor.shape}")
print(f"  X_test_tensor:  {X_test_tensor.shape}")

## 8. Define LSTM model

In [ ]:
class MixedSignalsLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, num_classes=3, dropout=0.2):
        super(MixedSignalsLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc1     = nn.Linear(hidden_size, 32)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2     = nn.Linear(32, num_classes)

    def forward(self, x):
        _, (hidden, _) = self.lstm(x)
        x = hidden[-1]          # final hidden state of last layer
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = MixedSignalsLSTM(input_size=n_features)
print(model)

## 9. Loss function & optimiser

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Criterion: Cross EntropyLoss")
print("Optimiser: Adam  lr=0.001")

## 10. Training loop

In [ ]:
epochs = 50

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    outputs = model(X_train_tensor)
    loss    = criterion(outputs, y_train_tensor)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        _, train_pred = torch.max(outputs, 1)
        train_acc = (train_pred == y_train_tensor).float().mean().item()
        print(f"Epoch [{epoch+1:2d}/{epochs}]  Loss: {loss.item():.4f}  Train Acc: {train_acc:.4f}")

## 11. Evaluate on test set

In [ ]:
model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, test_pred = torch.max(test_outputs, 1)

test_pred_np = test_pred.numpy()
y_test_np    = y_test_tensor.numpy()

print("=" * 60)
print("Test Set Performance")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y_test_np, test_pred_np):.4f}\n")
print(classification_report(y_test_np, test_pred_np, target_names=["SELL", "HOLD", "BUY"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test_np, test_pred_np))

## 12. Predict a new 30-day case
Example: a month-long history of a classic mixed-signal subject.

In [ ]:
new_case = np.array([
    [4, 30, 0.30, 0.20, 0.35, 0.70, 0.50, 1, 1.5, 7.5, 0.80],
    [3, 36, 0.20, 0.10, 0.25, 0.80, 0.60, 1, 1.2, 8.0, 0.85],
    [5, 24, 0.40, 0.25, 0.40, 0.65, 0.30, 0, 2.0, 6.8, 0.70],
    [2, 48, 0.10, 0.05, 0.20, 0.85, 0.70, 1, 1.0, 8.5, 0.90],
    [4, 40, 0.20, 0.10, 0.25, 0.78, 0.60, 1, 1.4, 7.8, 0.82],
    [3, 50, 0.10, 0.08, 0.15, 0.82, 0.80, 1, 0.8, 8.7, 0.88],
    [5, 20, 0.50, 0.30, 0.45, 0.60, 0.20, 0, 2.3, 6.0, 0.55],
    [4, 28, 0.40, 0.22, 0.35, 0.68, 0.35, 1, 1.8, 6.9, 0.65],
    [3, 38, 0.25, 0.12, 0.25, 0.75, 0.55, 1, 1.3, 7.9, 0.79],
    [2, 60, 0.05, 0.02, 0.10, 0.90, 0.85, 1, 0.5, 9.2, 0.95],
    [4, 32, 0.30, 0.18, 0.28, 0.72, 0.45, 1, 1.7, 7.2, 0.76],
    [5, 26, 0.35, 0.25, 0.38, 0.65, 0.30, 0, 2.1, 6.5, 0.60],
    [3, 44, 0.15, 0.10, 0.20, 0.80, 0.65, 1, 1.1, 8.2, 0.87],
    [4, 35, 0.25, 0.15, 0.28, 0.75, 0.55, 1, 1.6, 7.7, 0.81],
    [6, 18, 0.55, 0.35, 0.50, 0.50, 0.20, 0, 2.8, 5.5, 0.48],
    [3, 42, 0.18, 0.10, 0.22, 0.78, 0.60, 1, 1.2, 8.1, 0.84],
    [2, 55, 0.08, 0.04, 0.12, 0.88, 0.80, 1, 0.7, 9.0, 0.93],
    [4, 30, 0.28, 0.18, 0.30, 0.70, 0.48, 1, 1.5, 7.3, 0.78],
    [5, 22, 0.45, 0.28, 0.40, 0.58, 0.25, 0, 2.4, 6.2, 0.58],
    [3, 46, 0.12, 0.08, 0.18, 0.82, 0.70, 1, 1.0, 8.4, 0.89],
    [4, 34, 0.26, 0.16, 0.29, 0.73, 0.52, 1, 1.6, 7.6, 0.80],
    [5, 24, 0.42, 0.26, 0.37, 0.62, 0.28, 0, 2.2, 6.6, 0.62],
    [2, 58, 0.05, 0.03, 0.10, 0.91, 0.88, 1, 0.4, 9.4, 0.97],
    [3, 41, 0.18, 0.09, 0.20, 0.79, 0.62, 1, 1.1, 8.0, 0.86],
    [4, 29, 0.32, 0.20, 0.33, 0.68, 0.40, 1, 1.8, 7.0, 0.72],
    [5, 21, 0.48, 0.30, 0.44, 0.55, 0.22, 0, 2.5, 6.0, 0.54],
    [3, 47, 0.14, 0.07, 0.18, 0.83, 0.68, 1, 0.9, 8.6, 0.90],
    [4, 33, 0.27, 0.17, 0.30, 0.71, 0.50, 1, 1.5, 7.4, 0.79],
    [2, 62, 0.04, 0.01, 0.08, 0.94, 0.90, 1, 0.3, 9.7, 0.99],
    [4, 36, 0.22, 0.14, 0.26, 0.76, 0.58, 1, 1.3, 7.9, 0.83]
], dtype=float)

new_case_scaled = scaler.transform(new_case)
new_case_scaled = np.expand_dims(new_case_scaled, axis=0)
new_case_tensor = torch.tensor(new_case_scaled, dtype=torch.float32)

with torch.no_grad():
    pred_logits = model(new_case_tensor)
    pred_probs  = torch.softmax(pred_logits, dim=1).numpy()[0]
    pred_class  = np.argmax(pred_probs)

label_map = {0: "SELL (Red Flag)", 1: "HOLD (Yellow)⚠️", 2: "BUY (Green Flag)"}

print("=" * 60)
print("New 30 day case prediction")
print("=" * 60)
print(f"Recommendation : {label_map[pred_class]}")
print(f"SELL probability (Red Flag): {pred_probs[0]:.4f}")
print(f"HOLD probability (Yellow Flag): {pred_probs[1]:.4f}")
print(f"BUY  probability (Green Flag): {pred_probs[2]:.4f}")

---
### Feature reference

| Feature | Direction | Weight |
|---|---|---|
| `engagement_volume` | + bullish | 0.08 |
| `latency_hours` | − bearish | 2.0 |
| `execution_ratio` | + bullish | 2.2 |
| `initiative_factor` | + bullish | 1.5 |
| `sentiment_stability` | + bullish | 1.8 |
| `after_hours_activity` | − bearish | 1.0 |
| `default_rate` | − bearish | 2.0 |
| `passive_engagement` | − bearish | 0.8 |
| `disclosure_quality` | + bullish | 1.8 |
| `behavioral_beta` | − bearish | 1.8 |
| `guidance_delivery_spread` | − bearish | 2.2 |